# Process Raw Data

This takes the raw data in the Model/rawdata folder and splits it along the detected 'line' between images

In [ ]:
from fastai.vision.all import *
from PIL import ImageFilter, Image
import numpy as np
import re

# Set some processing variables

In [ ]:
training_width, training_height = 480,640 # portrait

# Set up util functions

In [ ]:
resize = Resize((training_height, training_width), 
                 resamples=(Image.BILINEAR, Image.NEAREST),
                 method=ResizeMethod.Pad,
                 pad_mode=PadMode.Zeros)

# process a single image
def process_image(img):
    # Do the resizing
    resize_img = resize(img, split_idx=1) # Resize the image to match our training.
    
    return resize_img.convert('RGB')

In [ ]:
# compile some regex for future matching

# These are for the data sets where the source has no tattoo and should be processed into a silhouette
## plain/silhouette naming pair convention 
source_re = re.compile(r'.*_source\..*') # for images without tattoos, but never had any
mask_re = re.compile(r'.*_mask\..*') # for manually created silhouettes for training

# Process the images

In [ ]:
# Fetch our files from the rawdata
raw_image_files = get_image_files('./data/silhouette/rawdata')

# loop over all the images and process them
for raw_image_file in raw_image_files:
    # load the image and process
    print(f"Processing {raw_image_file}")
    pil_image = PILImage.create(raw_image_file)
    resize_img = process_image(pil_image)
    
    # Save the file
    basename = os.path.basename(raw_image_file)
    if source_re.match(basename):
        resize_img.save("./data/silhouette/source/" + re.sub(r'_source', "", basename))
    elif mask_re.match(basename):
        resize_img.save("./data/silhouette/mask/" + re.sub(r'_mask', "", basename))
    else: 
        print(f"WARNING: {raw_image_file} does not match naming convention. Skipping.")